# Airbnb Stock Price Prediction

**Author:** Tajamul Khan  
**Task:** One-step-ahead time-series regression  
**Primary metric:** MAE

## Project introduction

Forecast Airbnb's next trading-day closing price from information available by the end of the current trading day. This is an educational forecasting example, not financial advice. A chronological holdout and shifted features prevent future-price leakage.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load and verify market data

In [ ]:
DATA_FILE = 'ABNB.csv'
PROJECT_FOLDER = 'Airbnb Stock Price Prediction'
DATA_SOURCE = 'https://www.kaggle.com/datasets/whenamancodes/airbnb-inc-stock-market-analysis'

def resolve_data_path(filename):
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "Supervised Learning Projects" / PROJECT_FOLDER / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} was not found. Place it beside the notebook. Dataset information: {DATA_SOURCE}"
    )

DATA_PATH = resolve_data_path(DATA_FILE)
raw = pd.read_csv(DATA_PATH, sep=',', low_memory=False)
print(f"Loaded {DATA_PATH.name}: {raw.shape[0]:,} rows × {raw.shape[1]} columns")

In [ ]:
required = {"Date", "Open", "High", "Low", "Close", "Volume"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Required columns are missing: {sorted(missing)}")
data = raw.copy()
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
data = data.dropna(subset=["Date", "Close"]).sort_values("Date").drop_duplicates("Date").reset_index(drop=True)
display(data.head())
display(data[["Open", "High", "Low", "Close", "Volume"]].describe().T.round(2))

In [ ]:
plt.figure(figsize=(12, 4))
sns.lineplot(data=data, x="Date", y="Close", color="#2563eb")
plt.title("Airbnb closing price over time")
plt.tight_layout()
plt.show()

## 3. Causal feature engineering

The target is tomorrow's close. Rolling features use only values available through the current row.

In [ ]:
data["target_close_next_day"] = data["Close"].shift(-1)
data["return_1d"] = data["Close"].pct_change()
data["intraday_range"] = data["High"] - data["Low"]
data["close_lag_1"] = data["Close"].shift(1)
data["close_lag_5"] = data["Close"].shift(5)
data["close_ma_5"] = data["Close"].rolling(5).mean()
data["close_ma_20"] = data["Close"].rolling(20).mean()
data["volume_ma_5"] = data["Volume"].rolling(5).mean()
model_data = data.dropna().reset_index(drop=True)

features = ["Open", "High", "Low", "Close", "Volume", "return_1d", "intraday_range", "close_lag_1", "close_lag_5", "close_ma_5", "close_ma_20", "volume_ma_5"]
split_at = int(len(model_data) * 0.80)
train, test = model_data.iloc[:split_at], model_data.iloc[split_at:]
X_train, y_train = train[features], train["target_close_next_day"]
X_test, y_test = test[features], test["target_close_next_day"]
print(f"Training through {train['Date'].max().date()} | Holdout from {test['Date'].min().date()}")

## 4. Time-series model comparison

In [ ]:
models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Ridge Regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1))]),
}
cv = TimeSeriesSplit(n_splits=5)
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1, error_score="raise")
    rows.append({"Model": name, "CV MAE": -scores["test_score"].mean(), "CV std": scores["test_score"].std()})
comparison = pd.DataFrame(rows).sort_values("CV MAE").reset_index(drop=True)
display(comparison.round(4))

best_name = comparison.loc[comparison["Model"] != "Median baseline", "Model"].iloc[0]
best_model = models[best_name].fit(X_train, y_train)

## 5. Chronological holdout evaluation

The previous close is a practical naive one-day forecast and is reported beside the selected model.

In [ ]:
predictions = best_model.predict(X_test)
naive_predictions = X_test["Close"].to_numpy()
results = pd.DataFrame({
    "Model": ["Previous-close baseline", best_name],
    "MAE": [mean_absolute_error(y_test, naive_predictions), mean_absolute_error(y_test, predictions)],
    "RMSE": [mean_squared_error(y_test, naive_predictions) ** 0.5, mean_squared_error(y_test, predictions) ** 0.5],
    "R²": [r2_score(y_test, naive_predictions), r2_score(y_test, predictions)],
})
display(results.round(4))

plot_data = test[["Date", "target_close_next_day"]].copy()
plot_data["prediction"] = predictions
plt.figure(figsize=(12, 4))
plt.plot(plot_data["Date"], plot_data["target_close_next_day"], label="Actual")
plt.plot(plot_data["Date"], plot_data["prediction"], label=best_name)
plt.title("One-day-ahead closing-price forecast on the holdout period")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Limitations and next steps

Stock prices are non-stationary and influenced by events absent from this dataset. Compare every model with the previous-close baseline, use walk-forward validation, include transaction costs for trading research, and never interpret a historical fit as a guaranteed return. All metrics are calculated at execution time.